## 1. Dataset Structure Check

In [1]:
import os

def count_images(path):
    return len([
        f for f in os.listdir(path)
        if f.endswith((".jpg", ".png", ".jpeg"))
    ])

print("Dataset overview loaded from YAML:\n")
print(open("dataset.yaml").read())

train = count_images("dataset/train/images")
val = count_images("dataset/val/images")
test = count_images("dataset/test/images")

print(f"\nTrain: {train}")
print(f"Val:   {val}")
print(f"Test:  {test}")

Dataset overview loaded from YAML:

train: dataset/train/images
val: dataset/val/images
test: dataset/test/images

nc: 3

names:
  0: crack
  1: potholes
  2: wall_peeling

Train: 546
Val:   117
Test:  117


In [ ]:
import os
import time
import glob
import cv2
import numpy as np
import random
import shutil
import pandas as pd
from ultralytics import YOLO

# =========================
# 🚨 FORCE OFFLINE MODE
# =========================
os.environ["ULTRALYTICS_OFFLINE"] = "1"
os.environ["YOLO_OFFLINE"] = "1"

# also prevent hub checks
os.environ["NO_PROXY"] = "*"

PROJECT_DIR = "/home/achin/COS40007-Group/jenny/runs/detect/iter_road"

# =========================
# IOU FUNCTION
# =========================
def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2 - x1) * max(0, y2 - y1)

    a1 = (box1[2] - box1[0]) * (box1[3] - box1[1])
    a2 = (box2[2] - box2[0]) * (box2[3] - box2[1])

    return inter / (a1 + a2 - inter + 1e-6)

# =========================
# BUILD DATASET SUBSET
# =========================
IMG_DIR = "dataset/train/images"
LBL_DIR = "dataset/train/labels"

ALL_IMAGES = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.endswith((".jpg", ".png", ".jpeg"))
])

random.seed(42)
random.shuffle(ALL_IMAGES)


def build_subset_folder(images, size):

    subset = images[:size]
    base = f"dataset_incremental/iter_{size}"

    img_out = f"{base}/images"
    lbl_out = f"{base}/labels"

    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    valid = 0
    missing = 0

    for img in subset:
        label = img.rsplit(".", 1)[0] + ".txt"

        img_src = f"{IMG_DIR}/{img}"
        lbl_src = f"{LBL_DIR}/{label}"

        if not os.path.exists(img_src) or not os.path.exists(lbl_src):
            missing += 1
            continue

        shutil.copy(img_src, f"{img_out}/{img}")
        shutil.copy(lbl_src, f"{lbl_out}/{label}")
        valid += 1

    yaml_path = f"{base}/data.yaml"

    with open(yaml_path, "w") as f:
        f.write(f"""
path: {base}

train: images
val: ../../dataset/val/images
test: ../../dataset/test/images

nc: 3
names:
  0: crack
  1: potholes
  2: wall_peeling
""")

    print(f"\n[iter_{size}] valid={valid}, missing={missing}")
    return yaml_path, base


# =========================
# EVALUATION FUNCTION (OFFLINE SAFE)
# =========================
def evaluate(model_path, data_yaml, run_name, split="test"):

    model = YOLO(str(model_path))   # LOCAL ONLY (NO DOWNLOAD)

    metrics = model.val(
        data=data_yaml,
        split=split,
        plots=False,
        verbose=False
    )

    image_paths = glob.glob(f"dataset/{split}/images/*")

    all_ious = []

    for img_path in image_paths:

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"

        gt_boxes = []

        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    _, xc, yc, bw, bh = map(float, line.split())

                    x1 = (xc - bw/2) * w
                    y1 = (yc - bh/2) * h
                    x2 = (xc + bw/2) * w
                    y2 = (yc + bh/2) * h

                    gt_boxes.append([x1, y1, x2, y2])

        pred = model.predict(img_path, conf=0.25, verbose=False)[0]
        pred_boxes = pred.boxes.xyxy.cpu().numpy() if pred.boxes is not None else []

        for gt in gt_boxes:
            best = 0
            for pr in pred_boxes:
                best = max(best, iou(gt, pr))
            all_ious.append(best)

    return {
        "Run": run_name,
        "Split": split,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "Mean IoU": float(np.mean(all_ious)) if all_ious else 0.0
    }


# =========================
# ITERATIVE TRAINING
# =========================
sizes = [400, 450, 500]

val_results = []
test_results = []

# IMPORTANT: first model is local file
prev_weights = "yolo26s.pt"

for size in sizes:

    print(f"\n================ ITER {size} ================\n")

    subset_yaml, base_path = build_subset_folder(ALL_IMAGES, size)

    model = YOLO(prev_weights)   # 🚨 ALWAYS LOCAL FILE

    start = time.time()

    model.train(
        data=subset_yaml,
        epochs=10,
        patience=20,
        imgsz=640,
        batch=16,

        optimizer="AdamW",
        lr0=0.0015,
        lrf=0.01,
        weight_decay=1e-4,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        fliplr=0.5,
        flipud=0.1,
        mosaic=1.0,
        copy_paste=0.3,
        degrees=10,
        translate=0.1,

        project=PROJECT_DIR,
        name=f"iter_{size}",
        exist_ok=True,
        device=0,
        amp=True
    )

    end = time.time()

    best_path = os.path.join(PROJECT_DIR, f"iter_{size}", "weights", "best.pt")

    print("\nEvaluating validation + test...")

    full_yaml = "dataset.yaml"

    val_summary = evaluate(best_path, full_yaml, f"iter_{size}", "val")
    test_summary = evaluate(best_path, full_yaml, f"iter_{size}", "test")

    val_summary["Time(min)"] = (end - start) / 60
    test_summary["Time(min)"] = (end - start) / 60

    val_results.append(val_summary)
    test_results.append(test_summary)

    # =========================
    # NEXT ITER MODEL UPDATE
    # =========================
    prev_weights = best_path

    print(f"Next model → {prev_weights}")


# =========================
# FINAL RESULTS
# =========================
val_df = pd.DataFrame(val_results)
test_df = pd.DataFrame(test_results)

print("\n=== VALIDATION RESULTS ===")
print(val_df)

print("\n=== TEST RESULTS ===")
print(test_df)


================ ITER 400 ================


[iter_400] valid=400, missing=0
Ultralytics 8.4.56 🚀 Python-3.9.25 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_incremental/iter_400/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26s.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=iter_40

/home/achin/COS40007-Group/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 455.8±264.9 MB/s, size: 85.3 KB)
val: Scanning /home/achin/COS40007-Group/jenny/dataset/val/labels.cache... 117 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 117/117 5.2Mit/s 0.0s


/home/achin/COS40007-Group/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


optimizer: AdamW(lr=0.0015, momentum=0.937) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0001), 126 bias(decay=0.0)
Plotting labels to /home/achin/COS40007-Group/jenny/runs/detect/iter_road/iter_400/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/achin/COS40007-Group/jenny/runs/detect/iter_road/iter_400
Starting training for 10 epochs...
Closing dataloader mosaic


...

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 9.6it/s 0.8s0.2s
                   all        117        165       0.44       0.47      0.471      0.219
Speed: 0.5ms preprocess, 4.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Ultralytics 8.4.56 🚀 Python-3.9.25 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
YOLO26s summary (fused): 122 layers, 9,466,341 parameters, 0 gradients, 20.5 GFLOPs
val: Fast image access ✅ (ping: 0.5±0.1 ms, read: 43.3±44.3 MB/s, size: 41.7 KB)
val: Scanning /home/achin/COS40007-Group/jenny/dataset/test/labels.cache... 117 images, 18 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 117/117 19.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 8/8 8.1it/s 1.0s0.2s
                   all        117        148      0.534      0.502      0.481      0.218
Speed: 0.5ms preprocess, 2.0ms inference, 0.0ms loss, 0.2ms postpro

/home/achin/COS40007-Group/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 16 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(


optimizer: AdamW(lr=0.0015, momentum=0.937) with parameter groups 114 weight(decay=0.0), 126 weight(decay=0.0001), 126 bias(decay=0.0)
Plotting labels to /home/achin/COS40007-Group/jenny/runs/detect/iter_road/iter_450/labels.jpg... 
Image sizes 640 train, 640 val
Using 8 dataloader workers
Logging results to /home/achin/COS40007-Group/jenny/runs/detect/iter_road/iter_450
Starting training for 10 epochs...
Closing dataloader mosaic


/home/achin/COS40007-Group/.venv/lib64/python3.9/site-packages/torch/utils/data/dataloader.py:627: UserWarning: This DataLoader will create 8 worker processes in total. Our suggested max number of worker in current system is 4, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       1/10      4.88G      1.855      2.235    0.03777          2        640: 100% ━━━━━━━━━━━━ 29/29 4.5it/s 6.5s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 11.7it/s 0.3s.2s
                   all        117        165      0.186      0.236       0.15     0.0622

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       2/10       5.2G      1.766      2.217    0.03565          2        640: 100% ━━━━━━━━━━━━ 29/29 7.3it/s 4.0s0.1s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 9.1it/s 0.4s0.2s
                   all        117        165      0.492       0.16       0.15     0.0392

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
       3/10       5.2G      1.813      2.409    0.03598          2        640: 100% ━━━

## 2. IoU + Evaluation Function

In [12]:
from ultralytics import YOLO
import numpy as np
import cv2, glob
import os
os.environ["ULTRALYTICS_OFFLINE"] = "1"

def iou(box1, box2):
    x1 = max(box1[0], box2[0])
    y1 = max(box1[1], box2[1])
    x2 = min(box1[2], box2[2])
    y2 = min(box1[3], box2[3])

    inter = max(0, x2-x1) * max(0, y2-y1)
    a1 = (box1[2]-box1[0]) * (box1[3]-box1[1])
    a2 = (box2[2]-box2[0]) * (box2[3]-box2[1])

    return inter / (a1 + a2 - inter + 1e-6)


import os
import glob
import cv2
import numpy as np
from ultralytics import YOLO

def evaluate(model_path, data_yaml, run_name, split="test"):

    model = YOLO(model_path)

    # YOLO built-in evaluation (precision/recall/mAP + confusion matrix)
    metrics = model.val(
        data=data_yaml,
        split=split,
        plots=False,                 
        save_confusion_matrix=False,
        verbose=False
    )

    # -------------------------
    # IoU computation (custom)
    # -------------------------
    image_paths = glob.glob("dataset/test/images/*")

    all_ious = []

    for img_path in image_paths:

        img = cv2.imread(img_path)
        if img is None:
            continue

        h, w = img.shape[:2]

        label_path = img_path.replace("images", "labels").rsplit(".", 1)[0] + ".txt"

        gt_boxes = []

        # GT boxes (YOLO format → xyxy)
        if os.path.exists(label_path):
            with open(label_path) as f:
                for line in f:
                    cls, xc, yc, bw, bh = map(float, line.split())

                    x1 = (xc - bw / 2) * w
                    y1 = (yc - bh / 2) * h
                    x2 = (xc + bw / 2) * w
                    y2 = (yc + bh / 2) * h

                    gt_boxes.append([x1, y1, x2, y2])

        # Predictions
        pred = model.predict(img_path, conf=0.25, verbose=False)[0]
        pred_boxes = pred.boxes.xyxy.cpu().numpy()

        # IoU matching
        for gt in gt_boxes:

            best_iou = 0

            for pr in pred_boxes:
                x1 = max(gt[0], pr[0])
                y1 = max(gt[1], pr[1])
                x2 = min(gt[2], pr[2])
                y2 = min(gt[3], pr[3])

                inter = max(0, x2 - x1) * max(0, y2 - y1)

                area_gt = (gt[2] - gt[0]) * (gt[3] - gt[1])
                area_pr = (pr[2] - pr[0]) * (pr[3] - pr[1])

                union = area_gt + area_pr - inter

                iou = inter / union if union > 0 else 0

                best_iou = max(best_iou, iou)

            all_ious.append(best_iou)

    # -------------------------
    # return summary
    # -------------------------
    return {
        "Run": run_name,
        "Split": split,
        "Precision": metrics.box.mp,
        "Recall": metrics.box.mr,
        "mAP50": metrics.box.map50,
        "mAP50-95": metrics.box.map,
        "Mean IoU": float(np.mean(all_ious)) if len(all_ious) > 0 else 0.0
    }

In [13]:
import os, shutil, random
from ultralytics import settings
settings.update({"weights_dir": "./weights"})



random.seed(42)

IMG_DIR = "dataset/train/images"
LBL_DIR = "dataset/train/labels"

ALL_IMAGES = sorted([
    f for f in os.listdir(IMG_DIR)
    if f.endswith((".jpg", ".png", ".jpeg"))
])

random.shuffle(ALL_IMAGES)


def build_subset_folder(images, size):

    subset_name = f"iter_{size}"
    base = f"dataset_incremental/{subset_name}"

    img_out = os.path.join(base, "images")
    lbl_out = os.path.join(base, "labels")

    os.makedirs(img_out, exist_ok=True)
    os.makedirs(lbl_out, exist_ok=True)

    valid, missing = 0, 0

    selected = images[:size]

    for img in selected:

        label = img.rsplit(".", 1)[0] + ".txt"

        img_src = os.path.join(IMG_DIR, img)
        lbl_src = os.path.join(LBL_DIR, label)

        if not os.path.exists(img_src) or not os.path.exists(lbl_src):
            missing += 1
            continue

        shutil.copy(img_src, os.path.join(img_out, img))
        shutil.copy(lbl_src, os.path.join(lbl_out, label))
        valid += 1

    print(f"\n[{subset_name}] valid={valid}, missing={missing}")

    # FIXED YAML (IMPORTANT PART)
    yaml_path = os.path.join(base, "data.yaml")

    content = f"""path: {os.path.abspath(base)}
    train: images
    val: ../../dataset/val/images
    test: ../../dataset/test/images
    
    nc: 3
    names:
      0: crack
      1: potholes
      2: wall_peeling
    """
    
    with open(yaml_path, "w") as f:
        f.write(content)

    return yaml_path

## 3. Iterative Training Pipeline

In [15]:
import os
import time
from ultralytics import YOLO

os.environ["ULTRALYTICS_OFFLINE"] = "1"
os.environ["YOLO_OFFLINE"] = "1"
os.environ["WANDB_MODE"] = "offline"

sizes = [400, 450, 500]

val_results = []
test_results = []

# -------------------------
# STEP 0: initial model
# -------------------------
prev_weights = os.path.abspath("./yolo26s.pt")

for i, size in enumerate(sizes):

    print(f"\n================ ITER {size} ================\n")

    subset_yaml = build_subset_folder(ALL_IMAGES, size)

    # -------------------------
    # LOAD MODEL (IMPORTANT FIX)
    # -------------------------
    model = YOLO(prev_weights, task="detect")

    start = time.time()

    model.train(
        data=subset_yaml,
        epochs=10,
        patience=20,
        imgsz=640,
        batch=16,

        optimizer="AdamW",
        lr0=0.0015,
        lrf=0.01,
        warmup_epochs=3,
        weight_decay=1e-4,

        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        fliplr=0.5,
        flipud=0.1,
        mosaic=1.0,
        copy_paste=0.3,
        degrees=10,
        translate=0.1,

        project="runs/detect/iter_road",
        name=f"iter_{size}",
        exist_ok=True,
        save=True,
        plots=True,
        device=0,
        amp=True
    )

    end = time.time()

    best_path = os.path.abspath(
        f"runs/detect/iter_road/iter_{size}/weights/best.pt"
    )

    print("\nEvaluating validation + test...")

    full_yaml = "dataset.yaml"

    val_summary = evaluate(best_path, full_yaml, "val", f"iter_{size}")
    test_summary = evaluate(best_path, full_yaml, "test", f"iter_{size}")

    val_summary["Time(min)"] = (end - start) / 60
    test_summary["Time(min)"] = (end - start) / 60

    val_results.append(val_summary)
    test_results.append(test_summary)

    # -------------------------
    # 🔥 UPDATE FOR NEXT ITERATION
    # -------------------------
    prev_weights = best_path


================ ITER 400 ================


[iter_400] valid=400, missing=0
Ultralytics 8.4.56 🚀 Python-3.9.25 torch-2.8.0+cu128 CUDA:0 (NVIDIA A100-SXM4-80GB, 81153MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=dataset_incremental/iter_400/data.yaml, degrees=10, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=10, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.1, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0015, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/home/achin/COS40007-Group/jenny/yolo26s.pt, momentum=0.937, mosaic=1

RuntimeError: Dataset 'dataset_incremental/iter_400/data.yaml' error ❌ YAML syntax error in 'dataset_incremental/iter_400/data.yaml': mapping values are not allowed in this context
  in "<unicode string>", line 2, column 10
Verify YAML with https://ray.run/tools/yaml-formatter

## 4. Compare Validation vs Test

In [ ]:
val_df = pd.DataFrame(val_results)
test_df = pd.DataFrame(test_results)

print("\n=== VALIDATION RESULTS ===")
print(val_df.sort_values("mAP50-95", ascending=False))

print("\n=== TEST RESULTS ===")
print(test_df.sort_values("mAP50-95", ascending=False))

In [ ]:
print("\n=== VALIDATION SET RESULTS ===")
print(validation_df[[
    "Run",
    "Training Images",
    "Precision",
    "Recall",
    "mAP50",
    "mAP50-95"
]])

print("\n=== TEST SET RESULTS ===")
print(test_df[[
    "Run",
    "Training Images",
    "Precision",
    "Recall",
    "mAP50",
    "mAP50-95",
    "Mean IoU",
    "Time(min)"
]])

## 5. Plot Loss + Metrics Curve

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os

def plot_iter_curves(iter_sizes):

    plt.figure(figsize=(15, 6))

    # ---- LOSS CURVES ----
    plt.subplot(1, 2, 1)

    for size in iter_sizes:

        csv_path = f"runs/detect/iter_road/iter_{size}/results.csv"

        if not os.path.exists(csv_path):
            continue

        df = pd.read_csv(csv_path)

        plt.plot(df["epoch"], df["train/box_loss"], label=f"{size} box")
        plt.plot(df["epoch"], df["train/cls_loss"], linestyle="--", label=f"{size} cls")

    plt.title("Training Loss Across Iterations")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(fontsize=8)

    # ---- METRICS CURVES ----
    plt.subplot(1, 2, 2)

    for size in iter_sizes:

        csv_path = f"runs/detect/iter_road/iter_{size}/results.csv"

        if not os.path.exists(csv_path):
            continue

        df = pd.read_csv(csv_path)

        plt.plot(df["epoch"], df["metrics/mAP50(B)"], label=f"{size} mAP50")
        plt.plot(df["epoch"], df["metrics/mAP50-95(B)"], linestyle="--", label=f"{size} mAP50-95")

    plt.title("Validation Metrics Across Iterations")
    plt.xlabel("Epoch")
    plt.ylabel("Score")
    plt.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


# your iterations
iter_sizes = list(range(400, 501, 50))

plot_iter_curves(iter_sizes)

## 6. Confusion Matrix Display

In [ ]:
for size in sizes:
    path = f"runs/detect/iter_road/iter_{size}/confusion_matrix.png"
    if os.path.exists(path):
        print(f"Confusion matrix saved: {path}")

## 7. Ground Truth vs Prediction with IoU

In [ ]:
from ultralytics import YOLO
import glob
import os
import cv2
import matplotlib.pyplot as plt

# -------------------------
# LOAD MODEL
# -------------------------
model = YOLO("runs/detect/road_damage/unfrozen/weights/best.pt")


# -------------------------
# CLASS NAMES (from dataset.yaml)
# -------------------------
class_names = ["crack", "potholes", "wall_peeling"]


# -------------------------
# INFERENCE LOOP
# -------------------------
for img_path in glob.glob("dataset/test/images/*"):

    img = cv2.imread(img_path)
    if img is None:
        continue

    h, w = img.shape[:2]

    # -------------------------
    # PREDICTIONS
    # -------------------------
    pred = model.predict(img_path, conf=0.25, verbose=False)[0]
    pred_boxes = pred.boxes.xyxy.cpu().numpy()
    pred_cls = pred.boxes.cls.cpu().numpy().astype(int)

    # -------------------------
    # GROUND TRUTH
    # -------------------------
    label_path = (
        img_path.replace("images", "labels")
        .rsplit(".", 1)[0] + ".txt"
    )

    gt_boxes = []

    if os.path.exists(label_path):

        with open(label_path) as f:

            for line in f:

                cls, xc, yc, bw, bh = map(float, line.split())

                x1 = (xc - bw / 2) * w
                y1 = (yc - bh / 2) * h
                x2 = (xc + bw / 2) * w
                y2 = (yc + bh / 2) * h

                gt_boxes.append((int(cls), [x1, y1, x2, y2]))

    # -------------------------
    # DRAW PREDICTIONS (RED)
    # -------------------------
    for box, cls_id in zip(pred_boxes, pred_cls):

        x1, y1, x2, y2 = map(int, box)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 0, 255), 2)

        cv2.putText(
            img,
            f"P: {class_names[cls_id]}",
            (x1, max(20, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 0, 255),
            2
        )

    # -------------------------
    # DRAW GROUND TRUTH (GREEN) + IOU
    # -------------------------
    for gt_cls, gt_box in gt_boxes:

        x1, y1, x2, y2 = map(int, gt_box)

        cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)

        best_iou = 0

        for p in pred_boxes:
            best_iou = max(best_iou, iou(gt_box, p))

        cv2.putText(
            img,
            f"G: {class_names[gt_cls]} IoU={best_iou:.2f}",
            (x1, max(20, y1 - 10)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (255, 255, 0),
            2
        )

    # -------------------------
    # LEGEND (ONCE ONLY)
    # -------------------------
    cv2.putText(
        img,
        "Green = Ground Truth | Red = Prediction",
        (10, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2
    )

    # -------------------------
    # SHOW IMAGE
    # -------------------------
    plt.figure(figsize=(8, 8))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(os.path.basename(img_path))
    plt.axis("off")
    plt.show()